[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/25_flash_attention.ipynb)

# 🔴 Hard: Flash Attention (Tiled)

Implement **tiled attention with online softmax** — the core idea behind Flash Attention.

### Signature
```python
def flash_attention(Q, K, V, block_size=32) -> Tensor:
    # Q, K, V: (B, S, D)
    # Returns: (B, S, D) — same as standard attention
```

### Key Insight
Instead of materializing the full S×S attention matrix, process in blocks:
1. For each Q-block, iterate over K/V blocks
2. Use **online softmax**: track running `max` and `sum`
3. Rescale accumulator when max changes: `acc *= exp(old_max - new_max)`
4. Final: `output = acc / row_sum`

Must give **identical** results to standard softmax attention.

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.4 MB/s eta 0:00:00


In [5]:
import torch
import math
from torch_judge import hint
hint("flash_attention")


💡 Hint for Flash Attention (Tiled):
   Process Q in blocks. For each Q-block, iterate over K/V blocks. Use online softmax: track running max and sum, rescale accumulator when max changes. output = acc / row_sum.



In [ ]:
# def flash_attention(Q, K, V, block_size=32):
#     """
#     Flash Attention 的简化实现（分块注意力计算）

#     核心思想：通过分块计算 + 在线 Softmax，避免显式构建完整的 SxS 注意力矩阵，
#     将内存复杂度从 O(S²) 降低到 O(S)。

#     参数:
#         Q: (batch_size, seq_len, d_k) - 查询矩阵
#         K: (batch_size, seq_len, d_k) - 键矩阵
#         V: (batch_size, seq_len, d_k) - 值矩阵
#         block_size: int - 分块大小，默认 32

#     返回:
#         output: (batch_size, seq_len, d_k) - 注意力输出
#     """
#     B, S, D = Q.shape  # B: batch大小, S: 序列长度, D: 每个头的维度

#     # 初始化输出张量
#     output = torch.zeros_like(Q)

#     # ==================== 外循环：遍历 Query 块 ====================
#     # 每个 Query 块独立处理，最终拼接到输出
#     for i in range(0, S, block_size):
#         # 提取当前 Query 块
#         qi = Q[:, i:i+block_size]  # (B, bs_q, D)
#         bs_q = qi.shape[1]  # 当前块的实际大小（最后一块可能小于 block_size）

#         # ---------- 初始化当前 Query 块的统计量 ----------
#         # row_max: 每个查询位置当前看到的最大分数（用于数值稳定）
#         # 初始化为负无穷，因为尚未看到任何 Key
#         row_max = torch.full((B, bs_q, 1), float('-inf'), device=Q.device)

#         # row_sum: 每个查询位置的 exp(分数 - row_max) 之和（Softmax 分母）
#         # 初始化为 0
#         row_sum = torch.zeros(B, bs_q, 1, device=Q.device)

#         # acc: 累加器，存储 exp(分数 - row_max) * V 的加权和（Softmax 分子）
#         # 初始化为 0
#         acc = torch.zeros(B, bs_q, D, device=Q.device)

#         # ==================== 内循环：遍历 Key/Value 块 ====================
#         # 对所有 Key/Value 块进行扫描，逐步更新统计量
#         for j in range(0, S, block_size):
#             # 提取当前 Key 和 Value 块
#             kj = K[:, j:j+block_size]  # (B, bs_k, D)
#             vj = V[:, j:j+block_size]  # (B, bs_k, D)

#             # ---------- 步骤 1: 计算当前块的注意力分数 ----------
#             # scores = (Q_block @ K_block^T) / sqrt(D)
#             # 形状: (B, bs_q, bs_k)
#             scores = torch.bmm(qi, kj.transpose(1, 2)) / math.sqrt(D)

#             # ---------- 步骤 2: 获取当前块的最大值 ----------
#             # 每个查询位置在当前 Key 块中的最大分数
#             # 形状: (B, bs_q, 1)
#             block_max = scores.max(dim=-1, keepdim=True).values

#             # ---------- 步骤 3: 更新全局最大值 ----------
#             # 融合历史最大值和当前块最大值
#             # 这是在线 Softmax 的关键：最大值可能变大
#             new_max = torch.maximum(row_max, block_max)  # (B, bs_q, 1)

#             # ---------- 步骤 4: 计算修正因子（缩放历史值） ----------
#             # 当最大值变大时，需要缩放之前累加的 exp 值
#             #
#             # 数学推导：
#             # 旧值: exp(x - row_max)
#             # 新值: exp(x - new_max) = exp(x - row_max) * exp(row_max - new_max)
#             # 所以缩放因子 correction = exp(row_max - new_max)
#             #
#             # 如果 new_max == row_max: correction = 1（无需缩放）
#             # 如果 new_max > row_max: correction < 1（历史值被压缩）
#             correction = torch.exp(row_max - new_max)  # (B, bs_q, 1)

#             # ---------- 步骤 5: 计算当前块的 exp（数值稳定） ----------
#             # 用新的全局最大值归一化，确保 exp 的最大值为 1
#             # 这避免了 exp 溢出（当分数很大时）
#             exp_scores = torch.exp(scores - new_max)  # (B, bs_q, bs_k)

#             # ---------- 步骤 6: 更新累加器（分子） ----------
#             # acc_new = acc_old * correction + exp_scores @ V_block
#             #
#             # 第一项: 缩放历史的加权和（因为 max 变了）
#             # 第二项: 当前块的加权和
#             #
#             # 形状: (B, bs_q, bs_k) @ (B, bs_k, D) -> (B, bs_q, D)
#             acc = acc * correction + torch.bmm(exp_scores, vj)

#             # ---------- 步骤 7: 更新行和（分母） ----------
#             # row_sum_new = row_sum_old * correction + sum(exp_scores)
#             #
#             # 第一项: 缩放历史 exp 之和
#             # 第二项: 当前块 exp 之和（在 Key 维度上求和）
#             #
#             # 形状: (B, bs_q, bs_k) -> sum -> (B, bs_q, 1)
#             row_sum = row_sum * correction + exp_scores.sum(dim=-1, keepdim=True)

#             # ---------- 步骤 8: 更新最大值 ----------
#             # 将当前最大值设为新的全局最大值，供下一轮迭代使用
#             row_max = new_max

#         # ---------- 输出归一化 ----------
#         # 最终注意力输出 = acc / row_sum
#         # 这等同于: softmax(scores) @ V
#         #
#         # 因为 acc = sum(exp(scores - row_max) * V)
#         #     row_sum = sum(exp(scores - row_max))
#         #     所以 acc / row_sum = softmax(scores) @ V
#         #
#         # 形状: (B, bs_q, D) / (B, bs_q, 1) -> (B, bs_q, D)
#         output[:, i:i+block_size] = acc / row_sum

#     return output

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

def flash_attention(Q, K, V, block_size=32):
    # Process Q in blocks, iterate K/V blocks with online softmax
    pass

In [4]:
# 🧪 Debug
import math
Q, K, V = torch.randn(1, 8, 4), torch.randn(1, 8, 4), torch.randn(1, 8, 4)
out = flash_attention(Q, K, V, block_size=4)
scores = torch.bmm(Q, K.transpose(1,2)) / math.sqrt(4)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
print('Match:', torch.allclose(out, ref, atol=1e-4))

TypeError: allclose(): argument 'input' (position 1) must be Tensor, not NoneType

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('flash_attention')